# Capstone Projects and the GenAI Interview Bank

Notebook for **Chapter 14** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 What Makes a Portfolio Project Hireable

## 14.2 Project 1: A RAG Assistant with an Evaluation Harness

## 14.3 Project 2: A Text-to-SQL Analytics Assistant with Guardrails

## 14.4 Project 3: A Work-Automation Agent (Weekly Sales Report Analyst)

In [ ]:
import numpy as np, re
rng=np.random.default_rng(42)
rows=[]
for week,scale in (("w01",1.0),("w02",0.80)):
    for cat,base in (("grocery",30),("electronics",120),("clothing",60)):
        s=scale if cat=="clothing" else 1.0
        val=rng.lognormal(np.log(base*s),0.25,400)
        ret=rng.random(400)<(0.06 if (week=="w02" and cat=="clothing") else 0.01)
        rows+= [(week,cat,float(v),bool(r)) for v,r in zip(val,ret)]
def stats(week,cat):
    x=np.array([r[2] for r in rows if r[0]==week and r[1]==cat]); rt=[r[3] for r in rows if r[0]==week and r[1]==cat]
    return {"avg_order":round(float(x.mean()),1),"return%":round(100*sum(rt)/len(rt),2)}
table={c:{w:stats(w,c) for w in ("w01","w02")} for c in ("grocery","electronics","clothing")}
for c,v in table.items(): print(c,v)
def tool_compare(cat,thr=0.10):
    a,b=table[cat]["w01"]["avg_order"],table[cat]["w02"]["avg_order"]
    return {"category":cat,"old":a,"new":b,"change_pct":round((b-a)/a*100,1),"drop":(a-b)/a>thr}
def fake_model(findings):
    bad=[f for f in findings if f["drop"]]
    return "Drops: "+"; ".join(f"{f['category']} average order {f['old']:.1f} -> {f['new']:.1f} dollars ({f['change_pct']}%)" for f in bad) if bad else "No drops."
findings=[tool_compare(c) for c in table]
report=fake_model(findings); print(report)
allowed={f"{f[k]:.1f}" for f in findings for k in("old","new")}|{str(abs(f["change_pct"])) for f in findings}
nums=re.findall(r"(?<![A-Za-z0-9.])\d+(?:\.\d+)?",report)
print("unverified numbers:",[n for n in nums if n not in allowed])

grocery {'w01': {'avg_order': 30.8, 'return%': 2.0}, 'w02': {'avg_order': 31.1, 'return%': 1.5}}
electronics {'w01': {'avg_order': 122.8, 'return%': 0.75}, 'w02': {'avg_order': 123.0, 'return%': 0.25}}
clothing {'w01': {'avg_order': 59.6, 'return%': 0.5}, 'w02': {'avg_order': 49.6, 'return%': 5.25}}
Drops: clothing average order 59.6 -> 49.6 dollars (-16.8%)
unverified numbers: []


## 14.5 Acceptance Criteria as Code, and How to Present

In [ ]:
results={"recall@5":0.91,"faithfulness":0.88,"idk_correct":0.80,"p95_s":7.4,"cost_per_task":0.0042,"injection_blocked":19,"injection_total":20}
criteria=[("recall@5",lambda r:r["recall@5"]>=0.90),
          ("faithfulness",lambda r:r["faithfulness"]>=0.90),
          ("idk_correct",lambda r:r["idk_correct"]>=0.75),
          ("p95_s",lambda r:r["p95_s"]<=8.0),
          ("cost_per_task",lambda r:r["cost_per_task"]<=0.005),
          ("injection",lambda r:r["injection_blocked"]/r["injection_total"]>=0.95)]
failed=[n for n,f in criteria if not f(results)]
for n,f in criteria: print("%-14s %s"%(n,"PASS" if f(results) else "FAIL"))
print("ACCEPTED" if not failed else "REJECTED: "+", ".join(failed))

recall@5       PASS
faithfulness   FAIL
idk_correct    PASS
p95_s          PASS
cost_per_task  PASS
injection      PASS
REJECTED: faithfulness


## 14.6 The GenAI Interview Bank

## Chapter recap

## Practice set

**1.** (Easy) A rule says "recall@5 at least 0.90". Your result is 0.88. Accepted?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch14.html